In [1]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

In [2]:
from config import (
    BEST_HUMAN_CUES_PATH,
    SEED_EXAMPLES_PATH,
    CATEGORY_ROUND0_SUMMARY_PATH,
    CATEGORY_APE_ALL_RESULTS_PATH,
    CATEGORY_APE_ALL_SUMMARY_PATH,
)
from ape_utils import (
    generate_outputs_for_prompt_df,
    summarize_prompt_scores,
    generate_resampled_prompt_df,
)

Using local model path: C:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Generator\models\Qwen2.5-1.5B-Instruct


`torch_dtype` is deprecated! Use `dtype` instead!


In [3]:
best_human_df = pd.read_csv(BEST_HUMAN_CUES_PATH)
seed_df = pd.read_csv(SEED_EXAMPLES_PATH)
round0_summary = pd.read_csv(CATEGORY_ROUND0_SUMMARY_PATH)

eval_df = best_human_df.copy()

In [4]:
TOP_K_PER_CATEGORY = 2
N_ROUNDS = 3
N_NEW_PROMPTS_PER_CATEGORY = 2

selected_prompt_df = (
    round0_summary
    .sort_values(["subcategory", "mean_proxy_score"], ascending=[True, False])
    .groupby("subcategory", as_index=False)
    .head(TOP_K_PER_CATEGORY)
    [["subcategory", "prompt_id", "prompt_family", "prompt_text"]]
    .copy()
)

selected_prompt_df["parent_prompt_id"] = ""

all_results = []
all_summaries = []

In [5]:
for ape_round in range(1, N_ROUNDS + 1):
    print(f"\n========== CATEGORY APE ROUND {ape_round} ==========")

    resampled_prompt_df = generate_resampled_prompt_df(
        top_prompt_summary_df=selected_prompt_df,
        n_new_prompts=N_NEW_PROMPTS_PER_CATEGORY,
        ape_round=ape_round,
    )

    print("Resampled prompts:")
    display(resampled_prompt_df.head(20))

    if resampled_prompt_df.empty:
        print("No new category prompts generated. Stop.")
        break

    round_results_df = generate_outputs_for_prompt_df(
        prompt_df=resampled_prompt_df,
        eval_df=eval_df,
        seed_df=seed_df,
        ape_round=ape_round,
    )

    round_summary_df = summarize_prompt_scores(round_results_df)

    if round_summary_df.empty:
        print("Round summary empty. Stop.")
        break

    round_summary_df["selection_status"] = "discarded"

    selected_rows = (
        round_summary_df
        .sort_values(["subcategory", "mean_proxy_score"], ascending=[True, False])
        .groupby("subcategory", as_index=False)
        .head(TOP_K_PER_CATEGORY)
        .index
    )

    round_summary_df.loc[selected_rows, "selection_status"] = "selected"

    display(round_summary_df.head(30))

    selected_prompt_df = round_summary_df[round_summary_df["selection_status"] == "selected"][
        ["subcategory", "prompt_id", "prompt_family", "prompt_text", "parent_prompt_id"]
    ].copy()

    all_results.append(round_results_df)
    all_summaries.append(round_summary_df)


========== CATEGORY APE ROUND 1 ==========
Resampled prompts:


,subcategory,prompt_id,prompt_family,prompt_text,parent_prompt_id
0,交通工具,交通工具_ape_1_28e0bada,category_resampled,您可以描述一下，當您看到一個固定的路線，有很多人一起搭乘時，您會如何感覺？這會讓您想起什麼特...,交通工具_p01
1,動作,動作_ape_1_b01dd8bd,category_resampled,**描述動作時的典型情境**：,動作_p03
2,動作,動作_ape_1_4fbdc65c,category_resampled,**目標詞彙**：吃早餐,動作_p03
3,動作,動作_ape_1_b57f0c6d,category_resampled,**提示句**：起床後，早晨的第一步，先洗漱，再吃早餐。,動作_p03
4,動作,動作_ape_1_fd3949e3,category_resampled,**描述動作的目的和工具**：,動作_p03
5,動作,動作_ape_1_185f0623,category_resampled,**目標詞彙**：刷牙,動作_p03
6,動作,動作_ape_1_0dad864b,category_resampled,**提示句**：早上起床後，開始新的一天，先刷牙，保持口腔清洁。,動作_p03
7,動物,動物_ape_1_7451dc86,category_resampled,**描述動物外觀與叫聲**：,動物_p02
8,動物,動物_ape_1_b50c4129,category_resampled,要點：描述外觀（如毛、耳朵、尾巴）、叫聲（如狗叫、貓咪叫）。,動物_p02
9,動物,動物_ape_1_115c4970,category_resampled,描述範圍：家裡、菜市場、公園。,動物_p02


APE round 1 prompts: 100%|██████████| 39/39 [06:48<00:00, 10.49s/it]


,ape_round,parent_prompt_id,subcategory,prompt_id,prompt_family,prompt_text,mean_proxy_score,mean_similarity,mean_brevity,mean_target_leak,selection_status
0,1,交通工具_p01,交通工具,交通工具_ape_1_28e0bada,category_resampled,您可以描述一下，當您看到一個固定的路線，有很多人一起搭乘時，您會如何感覺？這會讓您想起什麼特...,0.084490,0.049271,0.166667,0.833333,selected
1,1,動作_p03,動作,動作_ape_1_fd3949e3,category_resampled,**描述動作的目的和工具**：,0.535615,0.389580,0.876364,0.120000,selected
2,1,動作_p03,動作,動作_ape_1_0dad864b,category_resampled,**提示句**：早上起床後，開始新的一天，先刷牙，保持口腔清洁。,0.500105,0.338926,0.876190,0.120000,selected
3,1,動作_p03,動作,動作_ape_1_b57f0c6d,category_resampled,**提示句**：起床後，早晨的第一步，先洗漱，再吃早餐。,0.493899,0.328427,0.880000,0.120000,discarded
4,1,動作_p03,動作,動作_ape_1_b01dd8bd,category_resampled,**描述動作時的典型情境**：,0.481032,0.328746,0.836364,0.160000,discarded
5,1,動作_p03,動作,動作_ape_1_185f0623,category_resampled,**目標詞彙**：刷牙,0.454760,0.323942,0.760000,0.240000,discarded
6,1,動作_p03,動作,動作_ape_1_4fbdc65c,category_resampled,**目標詞彙**：吃早餐,0.418498,0.289283,0.720000,0.280000,discarded
7,1,動物_p02,動物,動物_ape_1_b50c4129,category_resampled,要點：描述外觀（如毛、耳朵、尾巴）、叫聲（如狗叫、貓咪叫）。,0.536204,0.337434,1.000000,0.000000,selected
8,1,動物_p02,動物,動物_ape_1_7451dc86,category_resampled,**描述動物外觀與叫聲**：,0.503418,0.376312,0.800000,0.200000,selected
9,1,動物_p02,動物,動物_ape_1_48bf7bbc,category_resampled,**描述動物生活環境與移動方式**：,0.470322,0.329031,0.800000,0.200000,discarded



========== CATEGORY APE ROUND 2 ==========
Resampled prompts:


...

,ape_round,parent_prompt_id,subcategory,prompt_id,prompt_family,prompt_text,mean_proxy_score,mean_similarity,mean_brevity,mean_target_leak,selection_status
0,2,動作_ape_1_fd3949e3,動作,動作_ape_2_b7c80da6,category_resampled,描述動作的目的和工具：,0.555249,0.416070,0.880000,0.120000,selected
1,2,動作_ape_1_fd3949e3,動作,動作_ape_2_ef6d0c83,category_resampled,提示句：早上起床後，開始新的一天，先刷牙，保持口腔清洁。,0.470069,0.311528,0.840000,0.160000,selected
2,2,動作_ape_1_fd3949e3,動作,動作_ape_2_7846ef34,category_resampled,提示句：早上起床後，開始新的一天，先刷牙，保持口腔潔凈。,0.456354,0.309894,0.798095,0.200000,discarded
3,2,動物_ape_1_b50c4129,動物,動物_ape_2_610d7833,category_resampled,描述叫聲（如狗叫、貓咪叫）。,0.528519,0.347884,0.950000,0.000000,selected
4,2,動物_ape_1_b50c4129,動物,動物_ape_2_13b96937,category_resampled,描述外觀（如毛、耳朵、尾巴）。,0.522509,0.403584,0.800000,0.200000,selected
5,2,喝_ape_1_1ba2a804,喝,喝_ape_2_70d5ed0b,category_resampled,請生成一句提示句，其中包含目標詞彙「柑橘類水果」，但不得直接說出「柑橘類水果」。,0.224139,0.177342,0.333333,0.666667,selected
6,2,喝_ape_1_1ba2a804,喝,喝_ape_2_b805d5b0,category_resampled,請生成一句提示句，其中包含目標詞彙「紫薯」，但不得直接說出「紫薯」。,0.000000,0.000000,0.000000,1.000000,selected
7,2,娛樂場所_ape_1_42e5edbc,娛樂場所,娛樂場所_ape_2_5f11db38,category_resampled,描述一種水果在自然環境中的生長過程。,0.000000,0.000000,0.000000,1.000000,selected
8,2,學校_ape_1_0cbec27c,學校,學校_ape_2_4f193443,category_resampled,為失語症患者生成一句具體、生活化且容易聯想到答案的語意提示句，請勿直接說出目標詞。,0.000000,0.000000,0.000000,1.000000,selected
9,2,家具_ape_1_cdf6dffd,家具,家具_ape_2_bdcdfec1,category_resampled,描述一種水果的種類，並說明其在夏天的適宜用途。,0.140954,0.094220,0.250000,0.750000,selected



========== CATEGORY APE ROUND 3 ==========
Resampled prompts:


,subcategory,prompt_id,prompt_family,prompt_text,parent_prompt_id
0,動作,動作_ape_3_6319ea44,category_resampled,"制作失語症治療提示句，以""早晨起床""作為起始，描述每天生活的開始。",動作_ape_2_b7c80da6
1,動作,動作_ape_3_d4fc740c,category_resampled,"適用於失語症治療提示，以""早上起床""作為起始，描述一天生活的開始。",動作_ape_2_b7c80da6
2,動物,動物_ape_3_1af21f7b,category_resampled,描述動物的叫聲（如狗叫、貓咪叫）。,動物_ape_2_610d7833
3,動物,動物_ape_3_2c03e45e,category_resampled,描述動物的外觀（如毛、耳朵、尾巴）。,動物_ape_2_610d7833
4,喝,喝_ape_3_6fc8c37d,category_resampled,請生成一句提示句，其中包含目標詞彙「橙子」，但不得直接說出「橙子」。,喝_ape_2_70d5ed0b
5,喝,喝_ape_3_b4952fdb,category_resampled,請生成一句提示句，其中包含目標詞彙「蘋果」，但不得直接說出「蘋果」。,喝_ape_2_70d5ed0b
6,娛樂場所,娛樂場所_ape_3_8b236995,category_resampled,描述一隻鳥如何在森林中繁殖。,娛樂場所_ape_2_5f11db38
7,娛樂場所,娛樂場所_ape_3_f43be8d6,category_resampled,描述一朵花如何在雨後開放。,娛樂場所_ape_2_5f11db38
8,學校,學校_ape_3_d696792a,category_resampled,為失語症患者生成一句具體、生活化且容易聯想到答案的語意提示句。,學校_ape_2_4f193443
9,學校,學校_ape_3_e7a8d43d,category_resampled,設計一組針對失語症患者的簡單、具體且容易聯想到答案的提示句列表。,學校_ape_2_4f193443


APE round 3 prompts: 100%|██████████| 29/29 [03:04<00:00,  6.35s/it]


,ape_round,parent_prompt_id,subcategory,prompt_id,prompt_family,prompt_text,mean_proxy_score,mean_similarity,mean_brevity,mean_target_leak,selection_status
0,3,動作_ape_2_b7c80da6,動作,動作_ape_3_6319ea44,category_resampled,"制作失語症治療提示句，以""早晨起床""作為起始，描述每天生活的開始。",0.463256,0.302610,0.838095,0.160000,selected
1,3,動作_ape_2_b7c80da6,動作,動作_ape_3_d4fc740c,category_resampled,"適用於失語症治療提示，以""早上起床""作為起始，描述一天生活的開始。",0.380159,0.237703,0.712554,0.280000,selected
2,3,動物_ape_2_610d7833,動物,動物_ape_3_2c03e45e,category_resampled,描述動物的外觀（如毛、耳朵、尾巴）。,0.594854,0.506934,0.800000,0.200000,selected
3,3,動物_ape_2_610d7833,動物,動物_ape_3_1af21f7b,category_resampled,描述動物的叫聲（如狗叫、貓咪叫）。,0.480733,0.258190,1.000000,0.000000,selected
4,3,喝_ape_2_70d5ed0b,喝,喝_ape_3_6fc8c37d,category_resampled,請生成一句提示句，其中包含目標詞彙「橙子」，但不得直接說出「橙子」。,0.000000,0.000000,0.000000,1.000000,selected
5,3,喝_ape_2_70d5ed0b,喝,喝_ape_3_b4952fdb,category_resampled,請生成一句提示句，其中包含目標詞彙「蘋果」，但不得直接說出「蘋果」。,0.000000,0.000000,0.000000,1.000000,selected
6,3,娛樂場所_ape_2_5f11db38,娛樂場所,娛樂場所_ape_3_8b236995,category_resampled,描述一隻鳥如何在森林中繁殖。,0.000000,0.000000,0.000000,1.000000,selected
7,3,娛樂場所_ape_2_5f11db38,娛樂場所,娛樂場所_ape_3_f43be8d6,category_resampled,描述一朵花如何在雨後開放。,0.000000,0.000000,0.000000,1.000000,selected
8,3,學校_ape_2_4f193443,學校,學校_ape_3_e7a8d43d,category_resampled,設計一組針對失語症患者的簡單、具體且容易聯想到答案的提示句列表。,0.635580,0.479400,1.000000,0.000000,selected
9,3,學校_ape_2_4f193443,學校,學校_ape_3_d696792a,category_resampled,為失語症患者生成一句具體、生活化且容易聯想到答案的語意提示句。,0.000000,0.000000,0.000000,1.000000,selected


In [6]:
all_results_df = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()
all_summaries_df = pd.concat(all_summaries, ignore_index=True) if all_summaries else pd.DataFrame()

all_results_df.to_csv(CATEGORY_APE_ALL_RESULTS_PATH, index=False, encoding="utf-8-sig")
all_summaries_df.to_csv(CATEGORY_APE_ALL_SUMMARY_PATH, index=False, encoding="utf-8-sig")

print("Saved:", CATEGORY_APE_ALL_RESULTS_PATH)
print("Saved:", CATEGORY_APE_ALL_SUMMARY_PATH)

Saved: C:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Generator\data\interim\category_ape_all_round_results.csv
Saved: C:\Users\804\Dtx_Projects\Aphasia_APE_Cue_Generator\data\interim\category_ape_all_round_prompt_summary.csv
